# Compare China e-commerce prices on Taobao, 1688, Xianyu and Douyin Shop

Search a product keyword on Taobao & Tmall (retail), 1688 (wholesale), Xianyu (second-hand) and Douyin Shop, and get one table of prices and sales, with the lowest, median and highest price on each platform and how many times the 1688 wholesale median the Taobao retail median is. No Taobao or 1688 account or Chinese phone number needed.

The script is [`monitor.py`](https://github.com/okdataapi/taobao-1688-price-monitor) (standard library only). It calls the [okdata](https://okdata.online/?src=colab-taobao-1688-price-monitor) API: a search page is $0.043 on Taobao, $0.022 on 1688 and Xianyu and $0.054 on Douyin Shop. One keyword on all four platforms is 4 calls, $0.141.

中文：输入一个商品关键词，同时搜淘宝、1688、闲鱼和抖音电商，看各平台的最低价、中位价和最高价，以及淘宝零售价是 1688 批发价的几倍。把下面的 `LANG` 改成 `"zh"`，报告就是中文。

## 1. Your API key

Get a key at [okdata.online](https://okdata.online/signup/?src=colab-taobao-1688-price-monitor) (free trial: up to 333 calls). In Colab, the easiest way is the key icon in the left sidebar (Secrets): add `OKDATA_API_KEY` and allow this notebook to read it. Otherwise the cell below asks for it. The key is kept in this session only.

In [ ]:
import os, getpass

def find_key():
    try:  # Colab secrets
        from google.colab import userdata
        return userdata.get("OKDATA_API_KEY")
    except Exception:
        pass
    try:  # Kaggle secrets
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("OKDATA_API_KEY")
    except Exception:
        pass
    return None

if not os.environ.get("OKDATA_API_KEY"):
    os.environ["OKDATA_API_KEY"] = find_key() or getpass.getpass("okdata API key (okd_...): ")
print("key loaded" if os.environ["OKDATA_API_KEY"].startswith("okd_") else "that does not look like an okdata key")

## 2. Get the script

In [ ]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/okdataapi/taobao-1688-price-monitor/main/monitor.py", "monitor.py")
print("monitor.py downloaded")

## 3. Choose what to monitor

Keywords as you would type them into the platform's search box (Chinese works best). `PLATFORMS` is any of `taobao`, `1688`, `xianyu`, `douyin_shop`, or `all`. `TRACK` follows particular products by `platform:id` or by the link you copy from the browser. `--dry-run` shows the plan and the most it can cost before anything is spent.

In [ ]:
import subprocess, sys

KEYWORDS = ["露营灯"]          # camping lantern; try 保温杯 (insulated cup), 蓝牙耳机 (earbuds)...
PLATFORMS = "all"              # or "taobao,1688" (the default, $0.065 per keyword)
TRACK = []                     # e.g. ["1688:594208233163", "https://detail.tmall.com/item.htm?id=827389194551"]
LANG = "en"                    # report language: en or zh

args = [a for k in KEYWORDS for a in ("--keyword", k)] + [a for t in TRACK for a in ("--track", t)]
args += ["--platforms", PLATFORMS, "--lang", LANG]

def run(*extra):
    r = subprocess.run([sys.executable, "monitor.py", *args, *extra], capture_output=True, text=True)
    print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)

run("--dry-run")

## 4. Take a snapshot

In [ ]:
run()

## 5. Look at the prices

In [ ]:
import pandas as pd

df = pd.read_csv("data/snapshots.csv", dtype={"product_id": str})
latest = df[(df.run_at == df.run_at.max()) & df.source.str.startswith("keyword:")]
latest.groupby(["source", "platform"]).price.describe()[["count", "min", "50%", "max"]].round(2)

In [ ]:
# The cheapest products of each platform, with what each platform adds: min_order on 1688,
# commission_rate on Douyin Shop, location (ships from / supplier / seller area).
cols = ["platform", "price", "sold", "shop", "location", "min_order", "commission_rate", "title", "url"]
latest.sort_values(["platform", "price"]).groupby("platform").head(5)[cols]

In [ ]:
import matplotlib.pyplot as plt

# Platform names in English, so the chart needs no Chinese font.
NAMES = {"taobao": "Taobao", "alibaba_1688": "1688", "xianyu": "Xianyu", "douyin_shop": "Douyin Shop"}
order = [p for p in NAMES if p in set(latest.platform)]
data = [latest[latest.platform == p].price.dropna() for p in order]
plt.figure(figsize=(8, 4.5))
plt.boxplot(data, showfliers=True)
plt.xticks(range(1, len(order) + 1), [NAMES[p] for p in order])
plt.yscale("log")
plt.ylabel("price, yuan (log scale)")
plt.title("Prices in the latest search results, by platform")
plt.tight_layout()
plt.show()

## 6. Run it again later

Come back tomorrow (or at least an hour later) and run the snapshot cell again: the report then lists the biggest price drops and the products that are new since the last run. A notebook session forgets its files when it ends, so for a lasting daily price history fork the [repository](https://github.com/okdataapi/taobao-1688-price-monitor) and let its GitHub Action take a snapshot every day.

The same data (product details with variants, reviews, shop listings) is also available to AI assistants such as Claude, Cursor and Dify through okdata's MCP server: see [okdata.online/mcp](https://okdata.online/mcp/?src=colab-taobao-1688-price-monitor).